# Stage 3.2: Hyperparameter Tuning with Optuna

This notebook covers the hyperparameter optimization phase of our project. 
Our objectives are to:
1. Load the preprocessed datasets and find the winning base configuration.
2. Utilize **Optuna** to optimize hyperparameter search space.
3. Define a custom objective target (`Recall + ROC-AUC`) to focus the study.
4. Retrain the final model on the chosen balanced dataset and evaluate on the test set.
5. Save the trained model artifact to `models/best_model.pkl`.
6. Inspect and business-interpret feature importances (top 15).

In [ ]:
import os
import sys
import joblib
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

# Add src folder to system path to import modules
sys.path.append('../src')
from evaluate import calculate_metrics, plot_confusion_matrix, plot_feature_importance
from train import load_processed_data, run_optuna_tuning

sns.set_theme(style="whitegrid")
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Load Processed Datasets

In [ ]:
X_train_smote, y_train_smote, X_train_smoteenn, y_train_smoteenn, X_test, y_test = load_processed_data('../data/processed')

# Load base model comparisons results to determine selection
base_results = pd.read_csv('../reports/base_model_comparisons.csv')
base_results['Selection_Score'] = base_results['Recall'] + base_results['ROC-AUC']
winner = base_results.sort_values(by='Selection_Score', ascending=False).iloc[0]

best_model_name = winner['Model']
best_balancing_name = winner['Balancing']
print(f"Tuning Selected Winner: {best_model_name} on {best_balancing_name} Balancing")

## 2. Set Up Datasets for Optimization
Extract training matrices corresponding to the best balancing technique.

In [ ]:
if best_balancing_name == 'SMOTE':
    X_train_best = X_train_smote
    y_train_best = y_train_smote
else:
    X_train_best = X_train_smoteenn
    y_train_best = y_train_smoteenn

print("Training matrices extracted successfully.")

## 3. Optuna Hyperparameter Optimization Study
We run 50 trials using Optuna's tree-structured Parzen Estimator (TPE) algorithm.

In [ ]:
best_params = run_optuna_tuning(
    best_model_name, best_balancing_name, X_train_best, y_train_best, X_test, y_test
)
print("\nTuned Hyperparameters:")
print(json.dumps(best_params, indent=2))

## 4. Final Model Retraining & Evaluation
We instantiate the model using our tuned parameters and retrain on the complete balanced training dataset.

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

if best_model_name == 'XGBoost':
    final_model = XGBClassifier(**best_params, random_state=42, eval_metric='logloss')
elif best_model_name == 'LightGBM':
    final_model = LGBMClassifier(**best_params, random_state=42, verbose=-1)
elif best_model_name == 'Random Forest':
    final_model = RandomForestClassifier(**best_params, random_state=42)
elif best_model_name == 'Logistic Regression':
    final_model = LogisticRegression(**best_params, random_state=42, max_iter=1000)
else:
    final_model = DecisionTreeClassifier(**best_params, random_state=42)

final_model.fit(X_train_best, y_train_best)

# Predict
y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1] if hasattr(final_model, 'predict_proba') else None

# Calculate final performance
final_metrics = calculate_metrics(y_test, y_pred, y_prob)
print("Final Performance Metrics:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.4f}")

## 5. Confusion Matrix Visualization

In [ ]:
plot_confusion_matrix(y_test, y_pred, f"Tuned {best_model_name}")

## 6. Feature Importances (Top 15)
We extract the top 15 features contributing to our predictions and plot their scores.

In [ ]:
feature_names = list(X_train_best.columns)
plot_feature_importance(final_model, feature_names, best_model_name, top_n=15)

## 7. Export Best Model
Save the finalized model binary file for prediction usage inside our Streamlit app.

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(final_model, '../models/best_model.pkl')
print("Model saved to models/best_model.pkl successfully!")